# 02 Baseline: TNG300-3 dark matter

Dark-matter density in, T-web classes out. Train, validation and test are separate slabs of the box, normalization is fit on the training slab, and the final scores come from the untouched test slab.

Runtime: **L4 or A100 GPU** with **High-RAM**. Expect one to two hours. If the session ends early, rerun the notebook from the top: the run resumes after its last completed epoch.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
!pip -q install --force-reinstall --no-deps "git+https://{token}@github.com/skumagai3/deepvoid.git"
del token
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
config = {
    'name': 'tng_dm_baseline_v2',
    'output': '/content/drive/MyDrive/deepvoid_runs',
    'seed': 12,
    'data': {'inputs': ['tng300-3/dm'], 'labels': 'tng300-3/mask'},
    'training': {'loss': {'name': 'scce', 'class_weights': 'auto'}},
}
# Everything else uses the defaults in docs/training.md (96^3 crops, AdamW,
# cosine schedule, EMA 0.999, early stopping on val_mcc with patience 15).

## 1. Watch progress

Refresh the TensorBoard panel while training runs. `log.csv` in the run folder on Drive updates every epoch too.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /tmp/deepvoid_tensorboard

## 2. Train, then score the test slab

In [ ]:
from deepvoid.train import run
out = run(config)

## 3. Results

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from deepvoid.labels import CLASS_NAMES

scores = json.load(open(out / 'test_scores.json'))['tng300-3/dm']
print(pd.DataFrame(scores['confusion_matrix'], index=CLASS_NAMES, columns=CLASS_NAMES))
print({k: round(v, 3) for k, v in scores.items() if k != 'confusion_matrix'})

log = pd.read_csv(out / 'log.csv')
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].plot(log.epoch, log.loss, label='train'); ax[0].plot(log.epoch, log.val_loss, label='val'); ax[0].set_title('loss'); ax[0].legend()
ax[1].plot(log.epoch, log.val_mcc, label='MCC'); ax[1].plot(log.epoch, log.val_void_f1, label='void F1'); ax[1].plot(log.epoch, log.val_halo_f1, label='halo F1'); ax[1].set_title('validation'); ax[1].legend()
ax[2].plot(log.epoch, log.learning_rate); ax[2].set_title('learning rate')
plt.tight_layout(); plt.show()

Paste the printed scores back to Claude.